<h3>test folium map layers</h3>

---

<h4>download an save data</h4>

In [ ]:
from variables import work_dir
from SOURCE import *
# import specific modules for copernicus
from SOURCE.data_source.copernicus import *

import copernicusmarine as cm

import datasets as dts
from variables import input_dir
from variables import output_dir
from variables import downld_directory as download_dir
from variables import downld_flag_bbox as flag_bbox
from variables import downld_targeted_bbox as targeted_bbox
from variables import downld_check_map as map
# uncomment the variable for bbox or poly
# bbox
#from variables import downls_bbox_geom as aoi_geom
# poly
from variables import downld_poly_geom as aoi_geom
from variables import downld_poly_edge as edge
#
from variables import map_flavours
from variables import max_num_downld_files as numberOfFiles
from variables import targeted_range

print("input !!")

print(input_dir)
print(output_dir)
print(cm.__version__)
print()

tools.checks.check_directory(input_dir)
tools.checks.check_directory(output_dir)
tools.checks.check_directory(download_dir)

# login to copernicus marine just one time
#cm.login()

# loop over datasets
for dataset, data in dts.datasets.items():
  print("dataset :",dataset)

  # downlod index files from copernicusmarine
  data_source.copernicus.utilities.download_index(input_dir,dataset,data)

  print("Ready !")

  info = data_source.copernicus.copernicus_functions.getIndexFilesInfo(
    data["details"],
    input_dir,
    targeted_bbox, 
    aoi_geom
  )
  #print(type(info))

In [ ]:
  import pandas as pd
  import json

  print("targeted range:",targeted_range)
  info['timeOverlap'] = info.apply(
    data_source.copernicus.copernicus_functions.timeOverlap,
    targeted_range=targeted_range,
    axis=1
  )
  #print(info.head())

  file_out = open('info_data.json','w')
  json.dump(pd.DataFrame.to_json(info), file_out)
  file_out.close()

In [ ]:
  # apply the filter
  subset = data_source.copernicus.utilities.filter_downloads(info,data["filters"])
  #print(type(subset))
  #print(subset.head())

  json_data = subset.to_json(orient='columns')

  with open('subset_data.json', 'w') as json_file:
    json.dump(json_data, json_file, indent=2)

In [ ]:
  # create download list
  data_source.copernicus.utilities.download_files_list(subset)

  # download data from copernicusmarine
  data_source.copernicus.utilities.download_data(dataset,download_dir)

  print("Done !!")

In [ ]:
  import pandas as pd
  import json
  from io import StringIO

  file_in = 'subset_data.json'
  with open(file_in) as input_data:
      load_data = json.load(input_data)

  subset = pd.read_json(StringIO(load_data))


In [ ]:
  # Check the position of the data:
#  map_check = tools.utilities.jupiter_create_map_checkdata(aoi_geom,edge,flag_bbox,map,numberOfFiles,subset,map_flavours[17])
#  map_check

print("Finish !")

---

<h4>read file and visualize data</h4>

<h4>--> --> restart kernel <--- <---</h4>

In [ ]:
from variables import work_dir
from SOURCE import *

from variables import output_dir
from variables import downld_directory as download_dir
from variables import downld_flag_bbox as flag_bbox
from variables import downld_targeted_bbox as targeted_bbox
from variables import downld_check_map as map
from variables import downld_poly_geom as aoi_geom
from variables import downld_poly_edge as edge

from variables import map_flavours
from variables import max_num_downld_files as numberOfFiles
from variables import targeted_range


In [ ]:
import pandas as pd
import json
from io import StringIO

file_in = 'subset_data.json'
with open(file_in) as input_data:
  load_data = json.load(input_data)

subset = pd.read_json(StringIO(load_data))


In [ ]:
import folium
import geopandas as gpd
import random
def popup_data(files,i):
  from datetime import datetime
  date_time = datetime.fromisoformat(files.iloc[i]['last_date_observation'].replace('Z', '+00:00'))
  date = date_time.strftime('%Y %b %d')
  time = date_time.strftime('%H:%M:%S')
  html = str(
    '<b>Platform code: </b><nowrap>' + str(files.iloc[i]['platform_code']) + '<br>' +
    '<b>Institution: </b><nowrap>' + files.iloc[i]['institution'] + '<br>' +
    '<b>Last Lat.: </b><nowrap>' + str(files.iloc[i]['last_latitude_observation']) + '<br>' +
    '<b>Last Lon.: </b><nowrap>' + str(files.iloc[i]['last_longitude_observation']) + '<br>' +
    '<b>Last Observation.: </b><nowrap>' + date + ' - ' + time + '<br>'
  )
  return html

def extract_unique_institution(subset):
  list = sorted(subset['institution'].unique().tolist())
  return list

colors = ['beige',
        'lightblue',
        'gray',
        'blue',
        'darkred',
        'lightgreen',
        'purple',
        'red',
        'green',
        'lightred',
        'darkblue',
        'darkpurple',
        'cadetblue',
        'orange',
        'pink',
        'lightgray',
        'darkgreen'
        'red',
        'lightred',
        'purple',
      ]

aoi_poly_geom = gpd.GeoDataFrame(index=[0], crs='epsg:4326', geometry=[aoi_geom]) 
edge = [
  [aoi_poly_geom.total_bounds[1],aoi_poly_geom.total_bounds[0]],
  [aoi_poly_geom.total_bounds[3],aoi_poly_geom.total_bounds[2]]
]

list_institution = extract_unique_institution(subset)

m = folium.Map(location=[map[0], map[1]], zoom_start=map[2])

folium.GeoJson(aoi_geom,color='orange',name='Area Of Interest').add_to(m)
folium.LatLngPopup().add_to(m)

for item in list_institution:
    df_subset = subset[subset["institution"] == item]
    fg = folium.FeatureGroup(name=item, show=True).add_to(m)
    for platform, files in df_subset.groupby(['institution', 'platform_code', 'data_type']):
        i = len(files)-1
        popup = folium.Popup(popup_data(files,i), min_width=150, max_width=400)
        idx = list_institution.index(files.iloc[i]['institution'])
        if idx <= len(colors):
            institution_idx = idx
        else:
            institution_idx = 2
        fg.add_child(folium.Marker([files.iloc[i]['last_latitude_observation'], files.iloc[i]['last_longitude_observation']], popup = popup, icon=folium.Icon(color=colors[institution_idx]) ))
    
folium.LayerControl().add_to(m)
#Zooming closer
m.fit_bounds(edge)

m


---